# Heterogeneous Beliefs Model of Stock Market Predictability

## Paper Citation
- **Title:** Heterogeneous Beliefs Model of Stock Market Predictability
- **Authors:** Jiho Park
- **Published:** 2024-06-12
- **ArXiv:** [https://arxiv.org/abs/2406.08448](https://arxiv.org/abs/2406.08448)

## Strategy Description
This notebook implements the strategy proposed in the paper by Jiho Park. The strategy is based on a model of heterogeneous beliefs where investors have different beliefs about the stochastic supply of assets. The model predicts momentum and reversal patterns in stock prices due to these heterogeneous beliefs.

The strategy will be implemented following the CRISP-TIQ 6-phase structure.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

### Configuration
In this phase, we define the universe of tickers, parameters, and the hypothesis of the strategy.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']  # Example universe, should be expanded to S&P 500 or subset
MOMENTUM_WINDOW = 126  # 6-month momentum window
REVERSAL_WINDOW = 252  # 12-month reversal window
POSITION_SIZING = 'equal_weight'  # Equal weight positions

# Hypothesis
# The strategy hypothesizes that stocks exhibiting momentum over a short window will continue to perform well,
# while stocks exhibiting momentum over a longer window will revert to their mean. This is based on the heterogeneous
# beliefs of investors about the stochastic supply of assets.

## Phase 2 — Data Download & Feature Computation

### Data Download
In this phase, we download historical market data for the defined universe of tickers and compute the necessary features (factors) for the strategy.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical market data
data = yf.download(UNIVERSE, start='2010-01-01', end='2024-06-12', group_by='ticker')

# Compute momentum and reversal signals
def compute_signals(data, momentum_window, reversal_window):
    momentum_signals = data['Adj Close'].pct_change(momentum_window).rank(axis=1, pct=True)
    reversal_signals = data['Adj Close'].pct_change(reversal_window).rank(axis=1, pct=True)
    return momentum_signals, reversal_signals

momentum_signals, reversal_signals = compute_signals(data, MOMENTUM_WINDOW, REVERSAL_WINDOW)

## Phase 3 — Signal Generation & Portfolio Construction

### Signal Generation
In this phase, we generate trading signals based on the computed features and construct the portfolio.

In [ ]:
# Generate trading signals
def generate_signals(momentum_signals, reversal_signals):
    long_signals = momentum_signals.where(momentum_signals > 0.5, 0)
    short_signals = reversal_signals.where(reversal_signals < 0.5, 0)
    return long_signals, short_signals

long_signals, short_signals = generate_signals(momentum_signals, reversal_signals)

# Construct portfolio
def construct_portfolio(long_signals, short_signals, position_sizing):
    if position_sizing == 'equal_weight':
        long_weights = long_signals / long_signals.sum(axis=1, skipna=True)
        short_weights = short_signals / short_signals.sum(axis=1, skipna=True)
    else:
        raise ValueError('Unsupported position sizing method')
    return long_weights, short_weights

long_weights, short_weights = construct_portfolio(long_signals, short_signals, POSITION_SIZING)

## Phase 4 — Vectorized Backtest

### Backtest
In this phase, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias by shifting signals forward by 1 period.

In [ ]:
# Perform vectorized backtest
def backtest(data, long_weights, short_weights):
    returns = data['Adj Close'].pct_change().dropna()
    long_returns = (returns * long_weights).sum(axis=1)
    short_returns = (returns * short_weights).sum(axis=1)
    strategy_returns = long_returns - short_returns
    return strategy_returns

strategy_returns = backtest(data, long_weights.shift(), short_weights.shift())

## Phase 5 — Performance Metrics

### Performance Metrics
In this phase, we calculate performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, max drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
def calculate_metrics(strategy_returns):
    cumulative_returns = (1 + strategy_returns).cumprod()
    max_drawdown = (cumulative_returns.cummax() - cumulative_returns) / cumulative_returns.cummax()
    sharpe_ratio = np.mean(strategy_returns) / np.std(strategy_returns)
    sortino_ratio = np.mean(strategy_returns) / np.std(strategy_returns[strategy_returns < 0])
    calmar_ratio = np.mean(strategy_returns) / max_drawdown.max()
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown, cumulative_returns

sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown, cumulative_returns = calculate_metrics(strategy_returns)

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns, label='Cumulative Returns')
plt.plot(max_drawdown, label='Max Drawdown')
plt.legend()
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown.max():.2f}')

## Phase 6 — Monitoring Stub

### Monitoring
In this phase, we create a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_portfolio(data, long_weights, short_weights):
    latest_returns = data['Adj Close'].pct_change().iloc[-1]
    long_pnl = (latest_returns * long_weights.iloc[-1]).sum()
    short_pnl = (latest_returns * short_weights.iloc[-1]).sum()
    total_pnl = long_pnl - short_pnl
    print(f'Daily P&L: {total_pnl:.2f}')
    print('Current Positions:')
    print('Long:', long_weights.columns[long_weights.iloc[-1] > 0].tolist())
    print('Short:', short_weights.columns[short_weights.iloc[-1] > 0].tolist())

# Example usage
monitor_portfolio(data, long_weights, short_weights)